In [1]:
pip install sdv pandas numpy

Note: you may need to restart the kernel to use updated packages.


## 1. Imports and basic configuration

In this cell we:

- Import **pandas** and **numpy** for data handling.
- Import **SingleTableMetadata** and **CTGANSynthesizer** from the SDV library.
- Set a random seed (`RANDOM_SEED`) so that our results are reproducible.

Line-by-line explanation:

- `import pandas as pd` – loads the pandas library and gives it the short name `pd`.
- `import numpy as np` – loads numpy for numerical operations.
- `from sdv.metadata import SingleTableMetadata` – class to describe our table structure (column types, primary key, etc.).
- `from sdv.single_table import CTGANSynthesizer` – GAN-based synthesizer that will learn from our real table and generate synthetic rows.
- `RANDOM_SEED = 42` – fixes the random seed so that each run uses the same random numbers.
- `np.random.seed(RANDOM_SEED)` – applies the seed to numpy’s random generator.


In [2]:
import pandas as pd
import numpy as np

from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

# For reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

## 2. Load the Kaggle financial fraud dataset (reference data)

In this cell we:

- Read the CSV file we downloaded from Kaggle into a pandas DataFrame.
- Inspect the shape of the data (number of rows and columns).
- Display the first few rows to understand the structure.

Line-by-line explanation:

- `file_path = "data/financial_fraud_data.csv"` – sets the relative path to the Kaggle CSV file. You should rename this to match your actual file name and folder.
- `df_real = pd.read_csv(file_path)` – reads the CSV file into a DataFrame called `df_real`.
- `print("Shape:", df_real.shape)` – prints the number of rows and columns in the DataFrame.
- `df_real.head()` – shows the first 5 rows so we can see the column names and example values.


In [3]:
file_path = "data/synthetic_fraud_dataset.csv"

df_real = pd.read_csv(file_path)

print("Shape:", df_real.shape)
df_real.head()

Shape: (10000, 10)


,transaction_id,user_id,amount,transaction_type,merchant_category,country,hour,device_risk_score,ip_risk_score,is_fraud
0,9608,363,4922.587542,ATM,Travel,TR,12,0.992347,0.947908,1
1,456,692,48.018303,QR,Food,US,21,0.168571,0.224057,0
2,4747,587,136.881960,Online,Travel,TR,14,0.296127,0.125058,0
3,6934,445,80.534719,POS,Clothing,TR,23,0.124801,0.159243,0
4,1646,729,120.041158,Online,Grocery,FR,16,0.098129,0.027542,0


## 3. Select columns for CTGAN and basic cleaning

CTGAN works best when we give it:

- Informative **numeric and categorical** features.
- We usually **drop pure IDs** (like transaction ID, customer ID) because they
  are high-cardinality identifiers and not useful as predictive features.

In this cell we:

1. Define a Python list called `columns_to_use` that contains the chosen feature names.
2. Create a smaller DataFrame `df_model` with only those columns.
3. Drop rows with missing values for simplicity (alternative: impute).
4. Print basic info to confirm.

Line-by-line explanation:

- `columns_to_use = [...]` – manually lists which columns we want to keep for training the generative model.
- `df_model = df_real[columns_to_use].copy()` – creates a new DataFrame with only the selected columns.
- `print("Before dropping NA:", df_model.shape)` – prints shape before cleaning.
- `df_model = df_model.dropna()` – drops rows that contain any missing values (simple cleaning choice).
- `print("After dropping NA:", df_model.shape)` – prints shape after cleaning so we see how many rows remain.
- `df_model.head()` – shows a preview of the cleaned feature set.


In [4]:
columns_to_use = [
    # numeric-type features
    "transaction_id",
    "user_id",
    "amount",
    "hour",
    "device_risk_score",
    "ip_risk_score",

    # categorical-type features
    "transaction_type",
    "merchant_category",
    "country",

    # fraud label
    "is_fraud"
]

df_model = df_real[columns_to_use].copy()

print("Before dropping NA:", df_model.shape)
df_model = df_model.dropna()
print("After dropping NA:", df_model.shape)

df_model.head()

Before dropping NA: (10000, 10)
After dropping NA: (10000, 10)


,transaction_id,user_id,amount,hour,device_risk_score,ip_risk_score,transaction_type,merchant_category,country,is_fraud
0,9608,363,4922.587542,12,0.992347,0.947908,ATM,Travel,TR,1
1,456,692,48.018303,21,0.168571,0.224057,QR,Food,US,0
2,4747,587,136.881960,14,0.296127,0.125058,Online,Travel,TR,0
3,6934,445,80.534719,23,0.124801,0.159243,POS,Clothing,TR,0
4,1646,729,120.041158,16,0.098129,0.027542,Online,Grocery,FR,0


## 4. Check fraud vs non-fraud distribution in reference data

This is not required for CTGAN, but it helps us understand:

- How imbalanced the dataset is.
- Whether CTGAN will see enough examples of fraud.

Line-by-line explanation:

- `label_col = "is_fraud"` – sets the name of the fraud label column.
- `df_model[label_col].value_counts()` – counts how many rows are fraud (1) and non-fraud (0).
- `.to_frame("count")` – converts the counts into a DataFrame and names the column "count".
- `.assign(percentage=lambda x: x["count"] / x["count"].sum() * 100)` – adds a new column `percentage` showing class proportions.


In [5]:
label_col = "is_fraud"

class_counts = (
    df_model[label_col]
    .value_counts()
    .to_frame("count")
    .assign(percentage=lambda x: x["count"] / x["count"].sum() * 100)
)

class_counts

,count,percentage
is_fraud,,
0,9500,95.0
1,500,5.0


## 5. Build SingleTableMetadata for the fraud table

SDV uses a metadata object to understand:

- Which columns are numerical vs categorical.
- Which column is the primary key (if any).
- Any additional constraints.

In this cell we:

1. Create a `SingleTableMetadata` object.
2. Ask it to automatically detect column types from our `df_model`.
3. (Optionally) set a primary key if we want one.

Line-by-line explanation:

- `metadata = SingleTableMetadata()` – creates an empty metadata object for a single table.
- `metadata.detect_from_dataframe(data=df_model)` – infers column types (numerical, categorical, boolean, etc.) from the DataFrame.
- `metadata.validate()` – checks that the inferred metadata is consistent and logs any issues.


In [6]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(data=df_model)

# Validate the inferred metadata (prints warnings if anything looks odd)
metadata.validate()

## 6. Train CTGANSynthesizer on the reference fraud data

Now we create a **CTGANSynthesizer**, which:

- Learns the joint distribution of all selected columns using a GAN.
- Handles both numerical and categorical columns.
- After training, can generate new synthetic rows that statistically resemble the real rows.

In this cell we:

1. Instantiate `CTGANSynthesizer` with the metadata.
2. Fit it on `df_model`.

Line-by-line explanation:

- `synthesizer = CTGANSynthesizer(metadata, epochs=100)` – creates a CTGAN model.
  - `metadata` tells the model about column types.
  - `epochs=100` means it will iterate 100 training cycles (you can increase for better fidelity if training time is acceptable).
- `synthesizer.fit(df_model)` – trains the CTGAN on our prepared fraud dataset.


In [7]:
synthesizer = CTGANSynthesizer(
    metadata,
    epochs=100,       # you can increase this if you want higher fidelity (but training time increases)
    verbose=True
)

synthesizer.fit(df_model)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sdv/single_table/base.py:168: FutureWarning:

The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/sdv/single_table/base.py:134: UserWarning:

We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.

Gen. (-1.23) | Discrim. (-0.09): 100%|██████████| 100/100 [03:46<00:00,  2.26s/it]


## 7. Generate synthetic financial transaction records

After fitting CTGAN, we can ask it to create new rows that are:

- Fully synthetic (no real row is copied).
- Statistically similar to the original data.

In this cell we:

1. Choose how many rows we want to generate (e.g. 3000).
2. Call `synthesizer.sample(num_rows=...)` to create a new DataFrame.
3. Inspect its shape and head.

Line-by-line explanation:

- `num_synthetic = 3000` – sets the number of synthetic rows to generate (must be ≥ 1000 to satisfy the assignment).
- `df_synth = synthesizer.sample(num_rows=num_synthetic)` – generates that many synthetic rows as a new DataFrame.
- `print("Synthetic shape:", df_synth.shape)` – prints the number of synthetic rows and columns.
- `df_synth.head()` – shows the first 5 synthetic rows for inspection.


In [8]:
num_synthetic = 3000  # >= 1000 as required by the spec

df_synth = synthesizer.sample(num_rows=num_synthetic)

print("Synthetic shape:", df_synth.shape)
df_synth.head()

Synthetic shape: (3000, 10)


,transaction_id,user_id,amount,hour,device_risk_score,ip_risk_score,transaction_type,merchant_category,country,is_fraud
0,5140469,953,93.227008,9,0.079682,0.271452,ATM,Travel,FR,0
1,9606001,189,77.854904,23,0.029611,0.050107,ATM,Electronics,US,0
2,577834,451,56.963147,16,0.013083,0.093800,Online,Clothing,UK,0
3,1192751,58,1.000000,12,0.174900,0.028995,QR,Electronics,US,0
4,16572070,50,31.650773,20,0.130275,0.130503,Online,Food,DE,0


## 8. Compare distributions: real vs synthetic (quick check)

Here we do simple checks:

- Compare basic statistics of numeric columns.
- Compare class distribution of `isFraud`.

These are not full evaluations, but they give an early sense of quality.

Line-by-line explanation:

- `df_model.describe()` – summary stats (mean, std, quartiles) for real data.
- `df_synth.describe()` – summary stats for synthetic data.
- Two blocks of `value_counts()` – class distribution for the fraud label in real vs synthetic data.


In [9]:
print("Real data summary:")
display(df_model.describe(include="all"))

print("\nSynthetic data summary:")
display(df_synth.describe(include="all"))

print("\nReal fraud distribution:")
print(df_model[label_col].value_counts(normalize=True))

print("\nSynthetic fraud distribution:")
print(df_synth[label_col].value_counts(normalize=True))

Real data summary:


,transaction_id,user_id,amount,hour,device_risk_score,ip_risk_score,transaction_type,merchant_category,country,is_fraud
count,10000.00000,10000.000000,10000.000000,10000.000000,10000.000000,10000.000000,10000,10000,10000,10000.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,4,5,6,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,POS,Food,US,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,2568,2023,2050,NaN
mean,4999.50000,500.058700,178.142763,14.247100,0.183773,0.184669,NaN,NaN,NaN,0.050000
std,2886.89568,288.328495,531.647950,5.347383,0.177381,0.175772,NaN,NaN,NaN,0.217956
min,0.00000,0.000000,1.000000,0.000000,0.000030,0.000009,NaN,NaN,NaN,0.000000
25%,2499.75000,247.000000,65.084753,10.000000,0.075721,0.077762,NaN,NaN,NaN,0.000000
50%,4999.50000,503.000000,101.686510,14.000000,0.156583,0.158290,NaN,NaN,NaN,0.000000
75%,7499.25000,750.250000,138.280872,19.000000,0.234939,0.236968,NaN,NaN,NaN,0.000000



Synthetic data summary:


,transaction_id,user_id,amount,hour,device_risk_score,ip_risk_score,transaction_type,merchant_category,country,is_fraud
count,3.000000e+03,3000.000000,3000.000000,3000.000000,3000.000000,3000.000000,3000,3000,3000,3000.000000
unique,NaN,NaN,NaN,NaN,NaN,NaN,4,5,6,NaN
top,NaN,NaN,NaN,NaN,NaN,NaN,QR,Electronics,US,NaN
freq,NaN,NaN,NaN,NaN,NaN,NaN,1080,904,788,NaN
mean,8.360571e+06,500.989333,233.677363,14.991667,0.216924,0.258897,NaN,NaN,NaN,0.156667
std,4.842912e+06,287.498608,539.231084,4.844594,0.253919,0.271164,NaN,NaN,NaN,0.363547
min,8.362000e+03,0.000000,1.000000,0.000000,0.000030,0.000009,NaN,NaN,NaN,0.000000
25%,4.160030e+06,253.750000,44.810609,12.000000,0.056302,0.105067,NaN,NaN,NaN,0.000000
50%,8.484802e+06,503.000000,74.040798,16.000000,0.122365,0.173286,NaN,NaN,NaN,0.000000
75%,1.247676e+07,747.250000,107.352787,18.000000,0.225383,0.250240,NaN,NaN,NaN,0.000000



Real fraud distribution:
is_fraud
0    0.95
1    0.05
Name: proportion, dtype: float64

Synthetic fraud distribution:
is_fraud
0    0.843333
1    0.156667
Name: proportion, dtype: float64


## 9. Save the synthetic fraud dataset to CSV

Finally, we save the generated synthetic dataset to a CSV file so it can be:

- Reused in other notebooks.
- Shared with team members.
- Referenced in the report as the **main working dataset**.

Line-by-line explanation:

- `output_path = "data/synthetic_fraud_ctgan.csv"` – sets output file name and folder.
- `df_synth.to_csv(output_path, index=False)` – writes the DataFrame to disk as a CSV file, without the index column.
- `print(...)` – confirms where the file was saved and how many rows it contains.


In [10]:
output_path = "data/synthetic_fraud_ctgan.csv"
df_synth.to_csv(output_path, index=False)

print(f"Synthetic dataset saved to: {output_path}")
print(f"Rows: {len(df_synth)}, Columns: {df_synth.shape[1]}")

Synthetic dataset saved to: data/synthetic_fraud_ctgan.csv
Rows: 3000, Columns: 10
